# Training Comparison

This notebook compares the four main training runs from Weights & Biases.

It pulls the same six metrics used in the ablation notebook and overlays them by epoch:

- `train/loss_v`
- `train/loss_l`
- `train/loss_weighted`
- `val/rmse`
- `val/valid`
- `val/match_rate`

The validation plots annotate the final value outside the axes so it is easy to compare end performance.


In [ ]:
from __future__ import annotations

import pandas as pd
import matplotlib.pyplot as plt
import wandb

ENTITY = "xanderbaatz-danmarks-tekniske-universitet-dtu"

RUNS = [
    {"project": "mp20_lattice_x0_em", "run_id": "14vv6rk4", "label": "x0 EM", "color": "#000000", "linestyle": "-"},
    {"project": "mp20_lattice_eps_em", "run_id": "7adsowdz", "label": "eps EM", "color": "#0072B2", "linestyle": "--"},
    {"project": "mp20_lattice_eps_pc", "run_id": "1w4iaf7m", "label": "eps PC", "color": "#D55E00", "linestyle": "-."},
    {"project": "mp20_lattice_x0_pc", "run_id": "i4pmd3cy", "label": "x0 PC", "color": "#009E73", "linestyle": ":"},
]

METRICS = [
    "train/loss_v",
    "train/loss_l",
    "train/loss_weighted",
    "val/rmse",
    "val/valid",
    "val/match_rate",
]

api = wandb.Api()

plt.rcParams["figure.figsize"] = (10, 5.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["font.size"] = 11


In [ ]:
def fetch_run_data(project: str, run_id: str) -> pd.DataFrame:
    run = api.run(f"{ENTITY}/{project}/{run_id}")
    history = run.history(keys=["epoch", *METRICS], pandas=True)
    history = history.dropna(how="all")
    history = history.sort_values("epoch").drop_duplicates(subset=["epoch"], keep="last")
    return history


all_histories = {}
for run in RUNS:
    run_key = f"{run['project']}/{run['run_id']}"
    print(f"Fetching {run['label']} from {run_key}")
    all_histories[run_key] = fetch_run_data(run["project"], run["run_id"])


In [ ]:
summary_rows = []
for run in RUNS:
    label = run["label"]
    run_key = f"{run['project']}/{run['run_id']}"
    history = all_histories.get(run_key)
    row = {"run": label}
    for metric in METRICS:
        frame = history[["epoch", metric]].dropna()
        row[metric] = None if frame.empty else float(frame.iloc[-1][metric])
        row["last_epoch"] = None if frame.empty else int(frame.iloc[-1]["epoch"])
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df


In [ ]:
def plot_metric(metric: str, title: str, annotate_last: bool = False) -> None:
    fig, ax = plt.subplots()
    endpoints = []

    for run in RUNS:
        run_key = f"{run['project']}/{run['run_id']}"
        history = all_histories[run_key][["epoch", metric]].dropna()
        if history.empty:
            continue

        ax.plot(
            history["epoch"],
            history[metric],
            label=run["label"],
            color=run["color"],
            linestyle=run["linestyle"],
            linewidth=2.4,
        )

        if annotate_last:
            last = history.iloc[-1]
            endpoints.append({
                "label": run["label"],
                "x": float(last["epoch"]),
                "y": float(last[metric]),
                "color": run["color"],
            })

    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric)
    ax.legend(frameon=True)

    if annotate_last and endpoints:
        endpoints = sorted(endpoints, key=lambda item: item["y"])
        y_min = min(item["y"] for item in endpoints)
        y_max = max(item["y"] for item in endpoints)
        span = max(y_max - y_min, 1e-6)
        gap = span * 0.08
        x_max = max(item["x"] for item in endpoints)
        x_pad = max(x_max * 0.06, 5.0)

        adjusted = []
        current = None
        for item in endpoints:
            target = item["y"]
            if current is None:
                current = target
            else:
                current = max(target, current + gap)
            adjusted.append(current)

        for item, y_text in zip(endpoints, adjusted):
            metric_name = metric.split("/")[-1].upper() if metric == "val/rmse" else metric.split("/")[-1].capitalize()
            ax.annotate(
                f"{metric_name}: {item['y']:.4f}",
                xy=(item["x"], item["y"]),
                xytext=(x_max + x_pad, y_text),
                color=item["color"],
                va="center",
                fontsize=10,
                arrowprops={"arrowstyle": "-", "color": item["color"], "lw": 1.2},
            )

        ax.set_xlim(right=x_max + x_pad * 2.0)

    plt.show()


In [ ]:
plot_metric('train/loss_v', 'Velocity Training Loss', annotate_last=False)


In [ ]:
plot_metric('train/loss_l', 'Lattice Training Loss', annotate_last=False)


In [ ]:
plot_metric('train/loss_weighted', 'Weighted Training Loss', annotate_last=False)


In [ ]:
plot_metric('val/rmse', 'Validation RMSE', annotate_last=True)


In [ ]:
plot_metric('val/valid', 'Validation Valid Rate', annotate_last=True)


In [ ]:
plot_metric('val/match_rate', 'Validation Match Rate', annotate_last=True)
